In [4]:
import os
import numpy as np


X_train = []
X_test = []
y_train = []
y_test = []

src_path = "/volatile/home/yb285618/sleep-stage-prediction/data/processed/dreamt"

for client_folder in os.listdir(src_path):
    file = os.path.join(src_path, client_folder)
    X_train.append(np.load(os.path.join(file, "train_data.npy"), allow_pickle=False))
    X_test.append(np.load(os.path.join(file, "test_data.npy"), allow_pickle=False))
    y_train.append(np.load(os.path.join(file, "train_target.npy"), allow_pickle=False))
    y_test.append(np.load(os.path.join(file, "test_target.npy"), allow_pickle=False))


In [5]:
print(len(y_train))

100


In [6]:
X_train = np.concat(X_train, axis=0)
y_train = np.concat(y_train, axis=0)
print(X_train.shape)

(2542669, 64, 7)


In [7]:
from sklearn.preprocessing import LabelEncoder

lb = LabelEncoder()
y_train_encoded = lb.fit_transform(y_train)

y_train_encoded

array([4, 1, 1, ..., 1, 4, 1], shape=(2542669,))

In [8]:
import torch
import torch.nn as nn


class CNN(nn.Module):
    def __init__(self, channel_in=12, kernel_size=7):
        super().__init__()
        # 64 * 12
        self.conv1 = nn.Conv1d(
            in_channels=channel_in,
            out_channels=128,
            kernel_size=kernel_size,
            stride=1,
            padding="same",
        )
        # => 64 * 64
        self.bn1 = nn.BatchNorm1d(128)
        self.bn2 = nn.BatchNorm1d(64)
        self.bn3 = nn.BatchNorm1d(32)
        self.bn4 = nn.BatchNorm1d(32)

        self.maxpool = nn.MaxPool1d(kernel_size=2)

        self.conv2 = nn.Conv1d(
            in_channels=128,
            out_channels=64,
            kernel_size=kernel_size,
            stride=1,
            padding="same",
        )

        self.conv3 = nn.Conv1d(
            in_channels=64,
            out_channels=32,
            kernel_size=kernel_size,
            stride=1,
            padding="same",
        )

        self.conv4 = nn.Conv1d(
            in_channels=32,
            out_channels=32,
            kernel_size=kernel_size,
            stride=1,
            padding="same",
        )
        # => 64 * 32

        self.fc1 = nn.Linear(32, 128)
        self.fc2 = nn.Linear(128, 5)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.conv2(x)
        x = self.bn2(x)
        x = self.relu(x)
        x = self.conv3(x)
        x = self.bn3(x)
        x = self.relu(x)
        x = self.conv4(x)
        x = self.bn4(x)
        x = self.relu(x)
        x = x.mean(dim=2)
        x = self.relu(self.fc1(x))
        return self.fc2(x)


In [15]:
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import Lambda, Compose

torch.manual_seed(42)

transform = Compose(
    [
        torch.FloatTensor,
        Lambda(lambda x: torch.permute(x, [1, 0])),
    ]
)


class DreamtDataset(Dataset):
    def __init__(self, X, y, transform=None, target_transform=None):
        super().__init__()
        self.X = X
        self.y = y
        self.transform = transform
        self.target_transform = target_transform

    def __getitem__(self, index):
        x = self.X[index]
        y = self.y[index]
        if self.transform:
            x = self.transform(self.X[index])

        if self.target_transform:
            y = self.target_transform(self.y[index])

        return x, y

    def __len__(self):
        return self.X.shape[0]


train_ds = DreamtDataset(X_train, y_train_encoded, transform=transform)
train_dl = DataLoader(train_ds, batch_size=128, shuffle=True)

In [49]:
sample, _ = next(iter(train_dl))

sample.dtype

torch.float32

In [16]:
from tqdm import tqdm

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def train_model(model, train_dl, epochs, lr=0.001, device="cpu"):
    optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9)
    criterion = nn.CrossEntropyLoss(reduction="sum")
    for epoch in tqdm(range(epochs)):
        model.train()
        empirical_risk = 0.0
        for X, y in train_dl:
            X = X.to(device)
            y = y.to(device)
            optimizer.zero_grad()
            pred = model(X)
            loss = criterion(pred, y)
            loss.backward()
            optimizer.step()
            empirical_risk += loss.item()

        empirical_risk /= len(train_dl.dataset)
        print("Train loss: %.3f" % (empirical_risk))


model = CNN(channel_in=7)
model.to(DEVICE)

train_model(model, train_dl, epochs=10, device=DEVICE)

 10%|█         | 1/10 [01:39<14:57, 99.78s/it]

Train loss: 0.918


 20%|██        | 2/10 [03:23<13:38, 102.37s/it]

Train loss: 0.789


 30%|███       | 3/10 [05:10<12:09, 104.15s/it]

...

Train loss: 0.671


 90%|█████████ | 9/10 [15:30<01:43, 103.31s/it]

Train loss: 0.664


100%|██████████| 10/10 [17:13<00:00, 103.37s/it]

Train loss: 0.658


In [17]:
from sklearn.metrics import f1_score, classification_report


def test_model(model, test_dl, device="cpu"):
    criterion = nn.CrossEntropyLoss(reduction="sum")
    model.eval()
    generalization_error = 0.0
    correct = 0
    all_preds = []
    all_targets = []

    with torch.no_grad():
        for X, y in test_dl:
            X = X.to(device)
            y = y.to(device)
            logits = model(X)
            loss = criterion(logits, y)
            pred = torch.argmax(logits, dim=1)
            correct += (pred == y).sum().item()
            generalization_error += loss.item()
            all_preds.append(pred.cpu())
            all_targets.append(y.cpu())

        y_pred = torch.cat(all_preds).numpy()
        y_true = torch.cat(all_targets).numpy()
        generalization_error /= len(test_dl.dataset)
        accuracy = correct / len(test_dl.dataset)
        print(
            "Generalization Error: %.3f, Accuracy %.3f"
            % (generalization_error, accuracy)
        )

    return y_true, y_pred, generalization_error, accuracy




In [18]:
X_test = np.concat(X_test, axis=0)
y_test = np.concat(y_test, axis=0)

y_test_encoded = lb.transform(y_test)

test_ds = DreamtDataset(X_test, lb.transform(y_test), transform=transform)
test_dl = DataLoader(test_ds, batch_size = 128)
y_true, y_pred, err, acc = test_model(model, test_dl, DEVICE)

print(f"erro {err} acc {acc}")

print(classification_report(y_true, y_pred, target_names = ["N1","N2","N3","R","W"]))

Generalization Error: 0.600, Accuracy 0.779
erro 0.5998440199114762 acc 0.779376921358426
              precision    recall  f1-score   support

          N1       0.56      0.11      0.18     53241
          N2       0.73      0.86      0.79    240614
          N3       0.69      0.63      0.66     16036
           R       0.72      0.70      0.71     50591
           W       0.85      0.86      0.86    275136

    accuracy                           0.78    635618
   macro avg       0.71      0.63      0.64    635618
weighted avg       0.77      0.78      0.76    635618



In [52]:
errors = []
accs = []
for client in range(len(X_test)):
    test_ds_client = DreamtDataset(
        X_test[client], lb.transform(y_test[client]), transform=transform
    )
    test_dl_client = DataLoader(test_ds_client, batch_size=128)
    y_true, y_pred, err, acc = test_model(model, test_dl_client, DEVICE)
    errors.append(err)
    accs.append(acc)

Generalization Error: 0.394, Accuracy 0.851
Generalization Error: 0.277, Accuracy 0.901
Generalization Error: 0.452, Accuracy 0.833
Generalization Error: 0.464, Accuracy 0.830
Generalization Error: 0.496, Accuracy 0.809
Generalization Error: 0.504, Accuracy 0.820
Generalization Error: 0.634, Accuracy 0.773
Generalization Error: 0.461, Accuracy 0.835
Generalization Error: 0.473, Accuracy 0.823
Generalization Error: 0.462, Accuracy 0.842
Generalization Error: 0.554, Accuracy 0.803
Generalization Error: 0.559, Accuracy 0.781
Generalization Error: 0.680, Accuracy 0.721
Generalization Error: 0.521, Accuracy 0.808
Generalization Error: 0.647, Accuracy 0.742
Generalization Error: 0.483, Accuracy 0.824
Generalization Error: 0.593, Accuracy 0.761
Generalization Error: 0.464, Accuracy 0.804
Generalization Error: 0.495, Accuracy 0.802
Generalization Error: 0.570, Accuracy 0.760
Generalization Error: 0.612, Accuracy 0.761


In [56]:
sum(errors) / len(accs)

0.5140776370308373

In [19]:
f1_score(y_true, y_pred, average="macro")

0.6395392825545191